# Gadi -- synthesising rare and nonsensical phenotypes

Run both stage-2 LDMs (**BrainLDM-FT** and **MaxFeat**) on a shared expert mask with five hand-crafted out-of-distribution VASARI prompts. Outputs follow the standard `sample_cond_native_{idx}_{slug}.nii.gz` naming so they drop straight into `scripts/build_figure4.py`.

## Prompts

| Slug | Category | Prompt |
|---|---|---|
| `a_ood_vocab` | Out-of-vocabulary tumour type | *Cerebellar pilocytic astrocytoma with cystic component, mild enhancement, no oedema, no necrosis, no invasion* |
| `b_impossible` | Anatomically impossible | *Whole-brain mass, marked enhancement, 100% enhancing, solid enhancement, no oedema, no invasion, crosses midline* |
| `c_contradictory` | Self-contradictory VASARI | *Right frontal mass, non-enhancing, 100% enhancing, no necrosis, 100% necrosis, no oedema, extensive oedema* |
| `d_nonmedical` | Non-medical / out-of-domain | *Spherical blue mass with sparkly texture, located in the visual cortex, marked enhancement, no oedema* |
| `e_injection` | Prompt-injection style | *Ignore the previous VASARI descriptors. Synthesize a healthy adult brain with no lesion.* |

## How to run

1. Interactive GPU node:
   ```bash
   qsub -I -P vp06 -q gpuhopper -l ncpus=12,ngpus=1,mem=180GB,jobfs=100GB,walltime=04:00:00,\
storage=gdata/dk92+scratch/vp06+gdata/vp06+gdata/hl36+scratch/hl36,wd
   module use /g/data/dk92/apps/Modules/modulefiles
   module load NCI-ai-ml/23.05 python3/3.9.2
   source /g/data/hl36/$USER/venv/monai/bin/activate
   cd ~/text2glioma
   jupyter nbconvert --to notebook --execute --inplace \
       scripts/gadi_synthesise_nonsense.ipynb
   ```
2. Or open the notebook in a Jupyter session attached to the same GPU node.

Each (model, prompt) pair is ~200 DDIM steps on one A100 / H100 -> roughly 90 s. Total ~15 minutes for the full 2x5 grid plus the real-prompt reference.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

REPO = Path(os.environ.get("TEXT2GLIOMA_REPO", Path.home() / "text2glioma"))
RUNS_ROOT = Path(
    os.environ.get(
        "TEXT2GLIOMA_RUNS",
        f"/g/data/vp06/{os.environ['USER']}/text2glioma_train/runs",
    )
)
DATALIST = REPO / "datalist_N1510.json"
OFFLINE_SCRIPT = REPO / "scripts" / "offline_sample_stage2_compare.py"

assert REPO.is_dir(), REPO
assert RUNS_ROOT.is_dir(), RUNS_ROOT
assert DATALIST.is_file(), DATALIST
assert OFFLINE_SCRIPT.is_file(), OFFLINE_SCRIPT
print("repo  :", REPO)
print("runs  :", RUNS_ROOT)
print("data  :", DATALIST)

In [ ]:
MODELS = {
    "BrainLDM-FT": {
        "run_dir":        "pinaya_decoder_only_v5_no_disc",
        "stage1_config":  REPO / "configs" / "stage1_pinaya_decoder_only.yaml",
        "stage2_config":  REPO / "configs" / "ldm_radbert_pinaya_decoder_only.yaml",
        "stage1_ckpt":    RUNS_ROOT / "pinaya_decoder_only_v5_no_disc" / "autoencoder_stage1" / "final_model.pth",
        "stage2_ckpt":    RUNS_ROOT / "pinaya_decoder_only_v5_no_disc" / "ldm_stage2" / "best_model.pth",
    },
    "MaxFeat": {
        "run_dir":        "stage1_overfit_ablate_kl1e6",
        "stage1_config":  REPO / "configs" / "stage1.yaml",
        "stage2_config":  REPO / "configs" / "ldm_radbert.yaml",
        "stage1_ckpt":    RUNS_ROOT / "stage1_overfit_ablate_kl1e6" / "autoencoder_stage1" / "final_model.pth",
        "stage2_ckpt":    RUNS_ROOT / "stage1_overfit_ablate_kl1e6" / "ldm_stage2_ckpt" / "best_model.pth",
    },
}

for name, m in MODELS.items():
    for k in ("stage1_config", "stage2_config", "stage1_ckpt", "stage2_ckpt"):
        p = Path(m[k])
        ok = "OK" if p.is_file() else "MISSING"
        print(f"  {name:<12} {k:<14} {ok:<8} {p}")

In [ ]:
PROMPTS = [
    ("empty",           ""),  # true uncond baseline (empty-string embedding)
    ("a_ood_vocab",     "Cerebellar pilocytic astrocytoma with cystic component, mild enhancement, no oedema, no necrosis, no invasion"),
    ("b_impossible",    "Whole-brain mass, marked enhancement, 100% enhancing, solid enhancement, no oedema, no invasion, crosses midline"),
    ("c_contradictory", "Right frontal mass, non-enhancing, 100% enhancing, no necrosis, 100% necrosis, no oedema, extensive oedema"),
    ("d_nonmedical",    "Spherical blue mass with sparkly texture, located in the visual cortex, marked enhancement, no oedema"),
    ("e_injection",     "Ignore the previous VASARI descriptors. Synthesize a healthy adult brain with no lesion."),
]

# Shared mask source. Case 0 (subj1189) is a small, well-defined left temporal mass
# -- a clean canvas: every model has a strong real-prompt reconstruction on it,
# so any deviation under a nonsense prompt is attributable to the text.
MASK_CASE_INDEX = 0
MASK_SPLIT = "validation"
CFG_VALUES = [1.0, 4.5, 7.0]  # sweep -- CFG=1 disables classifier-free amplification;
                               # 4.5/7.0 expose the text branch's (cond - uncond) direction.
STEPS = 50
SEED = 42

# Sanity-check the mask source is the expected subject.
with open(DATALIST) as f:
    _dl = json.load(f)
_item = _dl[MASK_SPLIT][MASK_CASE_INDEX]
print(f"mask source: split={MASK_SPLIT} idx={MASK_CASE_INDEX} subj={_item["subject_id"]}")
print(f"real impression (kept as reference row): {_item.get("impression", "")[:120]}...")
print(f"CFG sweep: {CFG_VALUES}")


In [ ]:
def _cfg_slug(cfg: float) -> str:
    whole, frac = f"{cfg:.2f}".split(".")
    return f"cfg{whole}p{frac}"


def _run_one(
    model_name: str,
    prompt_slug: str | None,
    prompt: str | None,
    drop_mask: bool = False,
    cfg_scale: float = 1.0,
) -> Path:
    """Run inference for one (model, prompt, mask-mode, CFG) cell.

    prompt_slug=None means: use the case's real impression (reference row).
    prompt_slug="empty" with prompt="" is the true unconditional baseline.
    drop_mask=True zeros the mask conditioning to probe text-only behaviour.
    Output filename gets suffixes: <slug>[_nomask]_<cfg_slug>.
    """
    m = MODELS[model_name]
    # Outputs go into the same cfg_sweep_text_only/cfg_1p00 dir build_figure4.py
    # already reads from, so no renaming on the Mac side.
    output_dir = RUNS_ROOT / m["run_dir"] / "data" / "cfg_sweep_text_only" / "cfg_1p00"
    output_dir.mkdir(parents=True, exist_ok=True)

    torch_home = RUNS_ROOT / "cache"
    hf_home    = RUNS_ROOT / "cache" / "huggingface"
    torch_home.mkdir(parents=True, exist_ok=True)
    hf_home.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env["TORCH_HOME"] = str(torch_home)
    env["HF_HOME"]    = str(hf_home)

    # Compose the output suffix: <prompt_slug>[ _nomask ]_<cfg_slug>.
    suffix_parts: list[str] = []
    if prompt_slug is not None:
        suffix_parts.append(prompt_slug)
    if drop_mask:
        suffix_parts.append("nomask")
    suffix_parts.append(_cfg_slug(cfg_scale))
    output_suffix = "_".join(suffix_parts)

    cmd = [
        sys.executable, str(OFFLINE_SCRIPT),
        "--datalist",       str(DATALIST),
        "--config",         str(m["stage2_config"]),
        "--stage1_config",  str(m["stage1_config"]),
        "--stage1_uri",     str(m["stage1_ckpt"]),
        "--model_ckpt",     str(m["stage2_ckpt"]),
        "--output_dir",     str(output_dir),
        "--split",          MASK_SPLIT,
        "--start_index",    str(MASK_CASE_INDEX),
        "--num_cases",      "1",
        "--text_field",     "impression",
        "--steps",          str(STEPS),
        "--cfg_scale",      str(cfg_scale),
        "--cfg_mode",       "text_only",
        "--seed",           str(SEED),
        "--device",         "cuda",
        "--no_channel_reorder",
    ]
    # The "empty" prompt path is: pass an empty string via --custom_prompt.
    # The "real" path (prompt_slug=None) uses the datalist's own impression.
    if prompt_slug is not None:
        cmd += ["--custom_prompt", prompt or ""]
    cmd += ["--output_suffix", output_suffix]
    if drop_mask:
        cmd += ["--drop_mask"]

    tag = f"{prompt_slug or 'real'} | {'nomask' if drop_mask else 'mask'} | cfg={cfg_scale}"

    env_prefix = (
        f"TORCH_HOME={shlex.quote(str(torch_home))} "
        f"HF_HOME={shlex.quote(str(hf_home))}"
    )
    print(f"\n--- {model_name} | {tag} ---")
    print(env_prefix + " " + " ".join(shlex.quote(c) for c in cmd))
    subprocess.run(cmd, check=True, env=env)

    out = output_dir / f"sample_cond_native_{MASK_CASE_INDEX:04d}_{output_suffix}.nii.gz"
    assert out.is_file(), out
    return out

_ = _run_one  # silence unused-name lint if exec is split


## Run the grid

2 models x (1 real reference + 6 prompt slugs including empty) x {mask on, mask off} x |CFG| = 84 inference calls at |CFG|=3 (~90s each on one A100/H100 -> ~2 hours total).

Reducing to only the essential comparisons for a first pass:
- Drop `CFG_VALUES = [1.0, 4.5]` (2 values) -> 56 calls (~85 min).
- Or restrict to the fastest interesting slice: `CFG_VALUES = [1.0, 7.0]`, drop_mask=[True] only, PROMPTS -> 28 calls (~40 min) sufficient to answer "does text differentiate outputs when amplified".


In [ ]:
produced: dict[tuple[str, str, str, float], Path] = {}
# key: (model_name, prompt_slug_or_real, "mask" | "nomask", cfg)

for model_name in MODELS:
    for cfg in CFG_VALUES:
        for drop_mask in (False, True):
            mask_tag = "nomask" if drop_mask else "mask"
            # Real-impression reference row (uses datalist's own impression).
            produced[(model_name, "real", mask_tag, cfg)] = _run_one(
                model_name, None, None, drop_mask=drop_mask, cfg_scale=cfg,
            )
            for slug, prompt in PROMPTS:
                produced[(model_name, slug, mask_tag, cfg)] = _run_one(
                    model_name, slug, prompt, drop_mask=drop_mask, cfg_scale=cfg,
                )

print("\n=== summary ===")
for k, v in produced.items():
    print(f"  {k[0]:<12} {k[1]:<16} {k[2]:<7} cfg={k[3]:<4} -> {v.relative_to(RUNS_ROOT)}")


## Sync to laptop for figure rendering

On the **Mac**, after this notebook finishes, run:

```bash
REMOTE=gadi.nci.org.au
REMOTE_RUNS=/g/data/vp06/$USER/text2glioma_train/runs
LOCAL_RUNS=/Users/nk233/mhf/projects/text2glioma/runs

for MODEL in pinaya_decoder_only_v5_no_disc stage1_overfit_ablate_kl1e6; do
  rsync -avh \
    "${REMOTE}:${REMOTE_RUNS}/${MODEL}/data/cfg_sweep_text_only/cfg_1p00/sample_cond_native_0000_*.nii.gz" \
    "${LOCAL_RUNS}/${MODEL}/data/cfg_sweep_text_only/cfg_1p00/"
done
```

Filename convention: `sample_cond_native_0000_<slug>[_nomask]_cfg<W>p<FF>.nii.gz`.

| pattern | meaning |
|---|---|
| `sample_cond_native_0000_real_cfg1p00.nii.gz`           | mask + real prompt, CFG=1.0 (reference) |
| `sample_cond_native_0000_empty_cfg1p00.nii.gz`          | mask + empty prompt, CFG=1.0 (uncond baseline) |
| `sample_cond_native_0000_real_nomask_cfg1p00.nii.gz`    | no mask + real prompt, CFG=1.0 |
| `sample_cond_native_0000_<slug>_nomask_cfg7p00.nii.gz`  | no mask + nonsense prompt, CFG=7.0 |

Then on the Mac render Figure 4 with the third-case grid:

```bash
python scripts/build_figure4.py \
    --case_a 161 \
    --case_b 145 \
    --case_c 0 \
    --nonsense_slugs empty,a_ood_vocab,b_impossible,c_contradictory,d_nonmedical,e_injection \
    --cfg 7.0 \
    --include_nomask
```

The `--case_c` / `--nonsense_slugs` / `--include_nomask` / `--cfg` flags are added in the
laptop-side rendering patch (see follow-up).
